In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, RandomFlip, RandomRotation, RandomZoom
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import os

# 1. 기본 설정
dataset_dir = 'desk_dataset' # 폴더 내에 'clean', 'dirty' 폴더가 있어야 합니다.
batch_size = 32
img_height = 224
img_width = 224
epochs = 200

# 2. 데이터셋 로드 및 분할 (학습용 80%, 검증용 20%)
train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

# 성능 최적화를 위한 prefetching
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

# 3. 데이터 증강 레이어 (Data Augmentation)
data_augmentation = Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.2),
    RandomZoom(0.2),
], name="data_augmentation")

# ResNet50 전처리 함수
preprocess_input = tf.keras.applications.resnet50.preprocess_input

# 4. 베이스 모델 로드 (ResNet50)
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))
base_model.trainable = False # 초기 학습 시에는 기존 가중치 동결

# 5. 모델 구축
inputs = tf.keras.Input(shape=(img_height, img_width, 3))
x = data_augmentation(inputs)  # 데이터 증강 적용 (학습 시에만 활성화됨)
x = preprocess_input(x)        # ResNet50 맞춤 전처리
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
outputs = Dense(1, activation='sigmoid')(x) # clean/dirty 이진 분류

model = Model(inputs, outputs)

# 6. 컴파일
model.compile(optimizer='adam', 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

class CustomPrintCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        loss = logs.get('loss')
        acc = logs.get('accuracy')
        val_loss = logs.get('val_loss')
        val_acc = logs.get('val_accuracy')
        
        print(f"\n\n=== [ 에포크 {epoch + 1} 학습 결과 ] ===")
        if loss is not None:
            print(f"▶ 학습 오차 (Loss)           : {loss:.4f} (학습 데이터와의 오차, 낮을수록 좋음 ⬇️)")
        if acc is not None:
            print(f"▶ 학습 정확도 (Accuracy)       : {acc:.4f} (학습 데이터 정답률, 높을수록 좋음 ⬆️)")
        if val_loss is not None:
            print(f"▶ 검증 오차 (Validation Loss)  : {val_loss:.4f} (새로운 데이터 오차, 실전 성능 지표 ⬇️)")
        if val_acc is not None:
            print(f"▶ 검증 정확도 (Val Accuracy)   : {val_acc:.4f} (새로운 데이터 정답률, 실전 성능 지표 ⬆️)")
        print("=================================\n")

# 7. 콜백 설정 (최적 모델 저장 및 조기 종료)
callbacks = [
    CustomPrintCallback(),
    ModelCheckpoint(
        filepath="desk_resnet50.keras",
        save_best_only=True,
        monitor='val_loss',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )
]

# 8. 모델 학습
print("학습을 시작합니다...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    callbacks=callbacks
)
print("학습이 완료되었습니다. 모델이 'desk_resnet50.keras'로 저장되었습니다.")

Found 1258 files belonging to 2 classes.
Using 1007 files for training.
Found 1258 files belonging to 2 classes.
Using 251 files for validation.
학습을 시작합니다...
Epoch 1/200
32/32 [==============================] - ETA: 0s - loss: 0.3023 - accuracy: 0.8888

=== [ 에포크 1 학습 결과 ] ===
▶ 학습 오차 (Loss)           : 0.3023 (학습 데이터와의 오차, 낮을수록 좋음 ⬇️)
▶ 학습 정확도 (Accuracy)       : 0.8888 (학습 데이터 정답률, 높을수록 좋음 ⬆️)
▶ 검증 오차 (Validation Loss)  : 0.2108 (새로운 데이터 오차, 실전 성능 지표 ⬇️)
▶ 검증 정확도 (Val Accuracy)   : 0.9124 (새로운 데이터 정답률, 실전 성능 지표 ⬆️)


Epoch 1: val_loss improved from inf to 0.21078, saving model to desk_resnet50.keras
32/32 [==============================] - 23s 161ms/step - loss: 0.3023 - accuracy: 0.8888 - val_loss: 0.2108 - val_accuracy: 0.9124
Epoch 2/200
31/32 [============================>.] - ETA: 0s - loss: 0.1025 - accuracy: 0.9627

=== [ 에포크 2 학습 결과 ] ===
▶ 학습 오차 (Loss)           : 0.1052 (학습 데이터와의 오차, 낮을수록 좋음 ⬇️)
▶ 학습 정확도 (Accuracy)       : 0.9623 (학습 데이터 정답률, 높을수록 좋음 ⬆️)
▶ 검증 오차 (Validation 

In [ ]:
import tensorflow as tf
import numpy as np
import os

# 1. 학습된 모델 불러오기
model_path = 'desk_resnet50.keras'
print("모델을 불러오는 중입니다...")
model = tf.keras.models.load_model(model_path)
print("모델 로드 완료!\n")

# 2. 클래스 이름 설정 (Keras는 폴더 이름 알파벳 순서로 0과 1을 부여합니다)
# C(clean)가 D(dirty)보다 먼저 오므로 clean=0, dirty=1 입니다.
class_names = ['clean', 'dirty']

# 3. 이미지 예측 함수 정의
def predict_desk_cleanliness(image_path):
    if not os.path.exists(image_path):
        print(f"오류: '{image_path}' 경로에 이미지가 없습니다.")
        return

    # 가. 이미지 불러오기 및 크기 조정 (학습 시 사용한 224x224 규격 유지)
    img = tf.keras.utils.load_img(image_path, target_size=(224, 224))
    
    # 나. 이미지를 Numpy 배열로 변환
    img_array = tf.keras.utils.img_to_array(img)
    
    # 다. 배치(Batch) 차원 추가 (모델은 여러 장을 동시에 처리하도록 설계되어 있어, 1장이라도 차원을 맞춰줘야 함)
    # (224, 224, 3) -> (1, 224, 224, 3)
    img_array = tf.expand_dims(img_array, 0)
    
    # 라. ResNet50 전처리 적용 (학습할 때와 동일한 전처리를 거쳐야 정확한 결과가 나옵니다)
    img_array = tf.keras.applications.resnet50.preprocess_input(img_array)
    
    # 마. 예측 수행
    predictions = model.predict(img_array, verbose=0)
    score = predictions[0][0] # Sigmoid 출력값이므로 0.0 ~ 1.0 사이의 값이 나옵니다.
    
    # 바. 결과 판별 (0.5를 기준으로 분류)
    if score > 0.5:
        result = class_names[1] # 0.5 초과면 dirty
        confidence = score * 100
    else:
        result = class_names[0] # 0.5 이하면 clean
        confidence = (1 - score) * 100
        
    print(f"▶ 테스트 이미지 : {os.path.basename(image_path)}")
    print(f"▶ 판별 결과     : [{result.upper()}]")
    print(f"▶ AI 확신도     : {confidence:.2f}%\n")

# 4. 실제 테스트 실행해보기
# 테스트해보고 싶은 이미지의 경로를 아래에 입력하세요.
test_image_1 = 'desk_test_1.jpg' # 예시 경로 (실제 경로로 수정 필요)
test_image_2 = 'desk_test_2.jpg' # 예시 경로 (실제 경로로 수정 필요)
test_image_3 = 'desk_test_3.jpg' # 예시 경로 (실제 경로로 수정 필요)
test_image_4 = 'desk_test_4.jpg' # 예시 경로 (실제 경로로 수정 필요)
test_image_5 = 'desk_test_5.jpg' # 예시 경로 (실제 경로로 수정 필요)
print("=== [ 책상 청결도 검증 테스트 ] ===")
predict_desk_cleanliness(test_image_1)
predict_desk_cleanliness(test_image_2)
predict_desk_cleanliness(test_image_3)
predict_desk_cleanliness(test_image_4)
predict_desk_cleanliness(test_image_5)

모델을 불러오는 중입니다...
모델 로드 완료!

=== [ 책상 청결도 검증 테스트 ] ===
▶ 테스트 이미지 : desk_test_1.jpg
▶ 판별 결과     : [CLEAN]
▶ AI 확신도     : 64.49%

▶ 테스트 이미지 : desk_test_2.jpg
▶ 판별 결과     : [DIRTY]
▶ AI 확신도     : 98.99%

▶ 테스트 이미지 : desk_test_3.jpg
▶ 판별 결과     : [CLEAN]
▶ AI 확신도     : 98.53%

▶ 테스트 이미지 : desk_test_4.jpg
▶ 판별 결과     : [CLEAN]
▶ AI 확신도     : 90.13%

▶ 테스트 이미지 : desk_test_5.jpg
▶ 판별 결과     : [DIRTY]
▶ AI 확신도     : 87.25%

